In [30]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, classification_report

In [14]:
df = pd.read_parquet("processed_ml_dataset.parquet")
data = pd.read_parquet("D:/TrendSpotting/TrendSpotting/download_dataset/data/openalex_corpus_1m.parquet")
df.head()

,doc_id,source_tier,novelty_raw,cluster_density,is_outlier_cluster,ppmi_domain_score,commercial_maturity_index,citation_velocity,citation_acceleration,distinct_venues_ratio,author_growth_rate,has_ref_data,target_emergence
0,W2280936323,journal,0.917789,0.102497,1,0.0,0.0,0.5000,0.200002,0.0,0.0,1,<NA>
1,W4232448486,other,0.916033,0.061456,0,0.0,NaN,0.0000,0.000000,0.0,0.0,1,<NA>
2,W2099050061,journal,0.320402,0.206780,0,0.0,NaN,0.2500,0.333336,0.0,0.0,1,<NA>
3,W2152433346,journal,0.738603,0.120838,0,0.0,0.0,4.4375,0.333334,0.0,0.0,1,<NA>
4,W1718665810,journal,0.890127,0.079078,0,0.0,0.0,1.1250,0.200002,0.0,0.0,1,<NA>


In [24]:
df = df.dropna(subset=["target_emergence"])
df.head()

,doc_id,source_tier,novelty_raw,cluster_density,is_outlier_cluster,ppmi_domain_score,commercial_maturity_index,citation_velocity,citation_acceleration,distinct_venues_ratio,author_growth_rate,has_ref_data,target_emergence
149311,W1970115948,journal,0.814098,0.072200,0,0.0,0.0,2.230769,1.000000,0.0,0.303682,1,0
149312,W2188122077,journal,0.783684,0.048243,0,0.0,0.0,0.076923,0.000000,0.0,-0.994623,1,0
149313,W2096533605,other,0.659670,0.049605,0,0.0,NaN,0.000000,0.000000,0.0,-0.125163,1,0
149314,W2066933239,journal,0.949024,0.195684,0,0.0,0.0,0.307692,0.333336,0.0,-0.719123,1,0
149315,W1536739630,other,0.613546,0.106075,0,0.0,NaN,0.076923,0.000000,0.0,0.174353,1,0


In [21]:
df["commercial_maturity_index"].isna().sum()

np.int64(139016)

In [22]:
df[df["commercial_maturity_index"].isna()][
    ["doc_id", "commercial_maturity_index"]
].head(20)

,doc_id,commercial_maturity_index
149313,W2096533605,NaN
149315,W1536739630,NaN
149323,W2144187926,NaN
149330,W1614633289,NaN
149335,W4231123991,NaN
149336,W2182122764,NaN
149342,W834014647,NaN
149347,W2799752344,NaN
149348,W2556095466,NaN
149354,W2037504719,NaN


In [23]:
print(df["target_emergence"].value_counts())
print(df["target_emergence"].value_counts(normalize=True))

target_emergence
0    545598
1     42638
Name: count, dtype: Int64
target_emergence
0    0.927515
1    0.072485
Name: proportion, dtype: Float64


In [27]:
X = df.drop(["target_emergence", "doc_id"], axis=1)
y = df.target_emergence

In [28]:
X.head()

,source_tier,novelty_raw,cluster_density,is_outlier_cluster,ppmi_domain_score,commercial_maturity_index,citation_velocity,citation_acceleration,distinct_venues_ratio,author_growth_rate,has_ref_data
149311,journal,0.814098,0.072200,0,0.0,0.0,2.230769,1.000000,0.0,0.303682,1
149312,journal,0.783684,0.048243,0,0.0,0.0,0.076923,0.000000,0.0,-0.994623,1
149313,other,0.659670,0.049605,0,0.0,NaN,0.000000,0.000000,0.0,-0.125163,1
149314,journal,0.949024,0.195684,0,0.0,0.0,0.307692,0.333336,0.0,-0.719123,1
149315,other,0.613546,0.106075,0,0.0,NaN,0.076923,0.000000,0.0,0.174353,1


In [29]:
y.head()

149311    0
149312    0
149313    0
149314    0
149315    0
Name: target_emergence, dtype: Int64

In [33]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [52]:
model = CatBoostClassifier(
    iterations=3000,
    depth=10,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="F1",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100
)

In [53]:
model.fit(
    X_train,
    y_train,
    cat_features=["source_tier"],
    eval_set=(X_test, y_test),
    early_stopping_rounds=100
)

0:	learn: 0.6665944	test: 0.6639887	best: 0.6639887 (0)	total: 164ms	remaining: 8m 12s
100:	learn: 0.7139145	test: 0.7071141	best: 0.7071326 (99)	total: 18.8s	remaining: 9m
200:	learn: 0.7350817	test: 0.7230441	best: 0.7230441 (200)	total: 43.4s	remaining: 10m 4s
300:	learn: 0.7613830	test: 0.7447269	best: 0.7447269 (300)	total: 1m 9s	remaining: 10m 23s
400:	learn: 0.7919818	test: 0.7721580	best: 0.7721580 (400)	total: 1m 35s	remaining: 10m 21s
500:	learn: 0.8125499	test: 0.7909350	best: 0.7909350 (500)	total: 2m 2s	remaining: 10m 9s
600:	learn: 0.8295449	test: 0.8072412	best: 0.8072412 (600)	total: 2m 28s	remaining: 9m 51s
700:	learn: 0.8438025	test: 0.8209513	best: 0.8210333 (699)	total: 2m 54s	remaining: 9m 31s
800:	learn: 0.8556803	test: 0.8303961	best: 0.8303961 (800)	total: 3m 20s	remaining: 9m 9s
900:	learn: 0.8667609	test: 0.8416102	best: 0.8416102 (900)	total: 3m 41s	remaining: 8m 35s
1000:	learn: 0.8766373	test: 0.8494282	best: 0.8494439 (999)	total: 3m 55s	remaining: 7m 51s


CatBoostClassifier(auto_class_weights='Balanced', depth=10, eval_metric='F1', iterations=3000, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [54]:
# Первый тест

pred = model.predict(X_test).astype(int).ravel()
proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, proba))

              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    109120
         1.0       0.49      0.90      0.64      8528

    accuracy                           0.93    117648
   macro avg       0.74      0.91      0.80    117648
weighted avg       0.96      0.93      0.93    117648

ROC-AUC: 0.9720270903153937


In [58]:
from catboost import Pool
import numpy as np
import pandas as pd

test_pool = Pool(
    X_test,
    cat_features=["source_tier"]
)

shap_values = model.get_feature_importance(
    test_pool,
    type="ShapValues"
)

# Последний столбец — базовое значение модели,
# поэтому его убираем
shap_features = shap_values[:, :-1]

In [59]:
shap_importance = pd.DataFrame({
    "feature": X_test.columns,
    "SHAP_importance": np.abs(shap_features).mean(axis=0)
})

shap_importance = shap_importance.sort_values(
    "SHAP_importance",
    ascending=False
)

print(shap_importance)

                      feature  SHAP_importance
9          author_growth_rate         0.864185
2             cluster_density         0.860812
4           ppmi_domain_score         0.207344
7       citation_acceleration         0.173421
6           citation_velocity         0.150221
1                 novelty_raw         0.121748
0                 source_tier         0.110458
5   commercial_maturity_index         0.053163
3          is_outlier_cluster         0.028934
8       distinct_venues_ratio         0.024824
10               has_ref_data         0.000000


In [57]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# Вероятности класса 1
proba = model.predict_proba(X_test)[:, 1]

thresholds = [0.85, 0.90, 0.92, 0.95, 0.9787982]

results = []

for threshold in thresholds:
    pred = (proba >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred)
    })

results_df = pd.DataFrame(results)

print(results_df)

   Threshold  Precision    Recall        F1
0   0.850000   0.828050  0.505394  0.627685
1   0.900000   0.890578  0.377932  0.530666
2   0.920000   0.910605  0.314142  0.467132
3   0.950000   0.948929  0.202627  0.333945
4   0.978798   0.981873  0.076220  0.141458


In [60]:
model.save_model("catboost_model.cbm")